# 02O — Meta-consistency & QC


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu, kruskal, spearmanr, ks_2samp, entropy, norm
from IPython.display import display

PROCESSED = Path('../../data/processed')
df = pd.read_csv(PROCESSED / 'DMD_variants_annotated.csv')

print('Loaded:', PROCESSED / 'DMD_variants_annotated.csv')
print('Shape:', df.shape)


Loaded: ../../data/processed/DMD_variants_annotated.csv
Shape: (11308, 30)


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for rel in ('../..', '..', '.'):
    cand = (Path.cwd() / rel).resolve()
    if (cand / 'src' / 'utils.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('project root with src/utils.py not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    add_consistency_flags,
    bh_adjust,
    chi2_table,
    ecdf_xy,
    fisher_bool,
    frame_mismatch,
    kruskal_group,
    mann_whitney_bool,
    mut_cons_mismatch,
    odds_ratio_ci,
    path_cons_mismatch,
    prepare_eda_dataframe,
    spearman_xy,
)

d = prepare_eda_dataframe(df)

print('Prepared shape:', d.shape)


Prepared shape: (11308, 59)


## 1. 📋 mismatch mutation vs consequence


In [3]:
tmp = d[['mutation_type', 'consequence']].copy()
tmp['mismatch_mut_cons'] = tmp.apply(mut_cons_mismatch, axis=1)
display(tmp['mismatch_mut_cons'].value_counts().to_frame('count'))
display(tmp[tmp['mismatch_mut_cons']].head(30))


,count
mismatch_mut_cons,
False,10391
True,917


,mutation_type,consequence,mismatch_mut_cons
193,splice,missense variant|intron variant,True
207,splice,intron variant,True
209,splice,intron variant,True
232,splice,intron variant,True
234,splice,intron variant,True
271,splice,synonymous variant,True
274,splice,intron variant,True
276,splice,intron variant,True
292,splice,intron variant,True
293,splice,intron variant,True


## 2. 📋 mismatch frame vs mutation


In [4]:
tmp = d[['mutation_type', 'frame']].copy()
tmp['mismatch_frame_mut'] = tmp.apply(frame_mismatch, axis=1)
display(tmp['mismatch_frame_mut'].value_counts().to_frame('count'))
display(tmp[tmp['mismatch_frame_mut']].head(30))


,count
mismatch_frame_mut,
False,11308


,mutation_type,frame,mismatch_frame_mut


## 3. 📋 mismatch pathogenic vs consequence


In [5]:
tmp = d[['pathogenic', 'consequence']].copy()
tmp['mismatch_path_cons'] = tmp.apply(path_cons_mismatch, axis=1)
display(tmp['mismatch_path_cons'].value_counts().to_frame('count'))
display(tmp[tmp['mismatch_path_cons']].head(30))


,count
mismatch_path_cons,
False,10887
True,421


,pathogenic,consequence,mismatch_path_cons
75,True,intron variant,True
142,False,3 prime UTR variant|frameshift variant,True
144,False,3 prime UTR variant|frameshift variant,True
148,False,3 prime UTR variant|frameshift variant,True
159,False,splice acceptor variant,True
187,True,splice donor variant|intron variant,True
188,False,splice donor variant|intron variant,True
189,False,splice donor variant|intron variant,True
195,False,nonsense|intron variant,True
235,False,splice donor variant,True


## 4. 📊 mismatch heatmap 📖


In [6]:
tmp = d[['var_id', 'mutation_type', 'consequence', 'frame', 'pathogenic']].copy()
tmp = add_consistency_flags(tmp)
heat = pd.DataFrame({
    'mismatch_mut_cons': [tmp['mismatch_mut_cons'].mean()],
    'mismatch_frame_mut': [tmp['mismatch_frame_mut'].mean()],
    'mismatch_path_cons': [tmp['mismatch_path_cons'].mean()]
})
fig = px.imshow(heat, aspect='auto', color_continuous_scale='Reds', title='Mismatch heatmap (fraction)')
fig.show()


## 5. 📋 inconsistent variants table 📖


In [7]:
tmp = d[['var_id', 'chr', 'pos', 'mutation_type', 'consequence', 'frame', 'pathogenic', 'phenotype', 'domain_clean', 'exon_num']].copy()
tmp = add_consistency_flags(tmp)
display(tmp[tmp['mismatch_any']].head(100))
display(pd.Series({'n_inconsistent_variants': int(tmp['mismatch_any'].sum())}).to_frame('value'))


,var_id,chr,pos,mutation_type,consequence,frame,pathogenic,phenotype,domain_clean,exon_num,mismatch_mut_cons,mismatch_frame_mut,mismatch_path_cons,mismatch_any
75,1327590,X,23617262.0,NaN,intron variant,NaN,True,BMD,NaN,NaN,False,False,True,True
142,4065957,X,31121853.0,frameshift,3 prime UTR variant|frameshift variant,out-of-frame,False,BMD,NaN,79.0,False,False,True,True
144,1190376,X,31121858.0,frameshift,3 prime UTR variant|frameshift variant,out-of-frame,False,NaN,NaN,79.0,False,False,True,True
148,201754,X,31121884.0,frameshift,3 prime UTR variant|frameshift variant,out-of-frame,False,DMD,Spectrin 2,79.0,False,False,True,True
159,4486967,X,31121932.0,splice,splice acceptor variant,out-of-frame,False,NaN,NaN,78.0,False,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
671,1322747,X,31173602.0,splice,splice acceptor variant|intron variant,out-of-frame,True,NaN,NaN,72.0,False,False,True,True
672,2690588,X,31173603.0,splice,splice acceptor variant|intron variant,out-of-frame,True,NaN,NaN,72.0,False,False,True,True
674,4487008,X,31173605.0,splice,splice acceptor variant|intron variant,out-of-frame,False,NaN,NaN,71.0,False,False,True,True
675,1655968,X,31173610.0,splice,intron variant,out-of-frame,False,DMD,NaN,71.0,True,False,False,True


,value
n_inconsistent_variants,1304
